In [1]:
import json
import numpy as np

In [4]:
data=json.load(open('/data/coding/object_graph.json','r'))

In [5]:
info=[]

for i in data:

    max_distance=0
    min_distance=10000
    max_label=-1
    min_label=1

    for j in object_list:

        if i==j:
            continue
    
        distance=np.linalg.norm(np.array(i['bbox_center'])-np.array(j['bbox_center']))
        if distance>max_distance:
            max_distance=distance 
            max_label=j['description']
        if distance<min_distance:
            min_distance=distance
            min_label=j['description'] 
        
    info.append({
        'id':i['id'],
        'label':i['description'],
        #'best_view':i['color_image_idx'],
        'max_distance':max_distance,
        'min_distance':min_distance,
        'max_label':max_label,
        'min_label':min_label
    })
        


NameError: name 'object_list' is not defined

In [23]:
label_list=[]         ## 唯一的label

for i in info:
    if i['label'] not in label_list:
        label_list.append(i['label'])

In [27]:
result={element: [] for element in label_list}

for i in info:
    result[i['label']].append(i)

In [6]:
prompt="Please refer to the following data:{}, which is in the format of \
{{'id ': object index,\
'label', Object category,\
'best_view': The best perspective of an object,\
'max_distance': The farthest distance,\
'min_distance': The closest distance,\
'max_label': The category of the farthest object,\
'min_label': The category of the object closest to each other}},\
answer the following question:{} directly (in lowercase), do not output any other information, do not output punctuation marks, and if it is a number, please output Arabic numerals".format(result,'Is there a car in this scene?')

completion = client.chat.completions.create(
    model="qwen-max-2025-01-25", # 此处以qwen-plus为例，可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/getting-started/models
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': prompt}],
    )
    
print(completion.choices[0].message.content)

NameError: name 'result' is not defined

In [7]:
import os
import cv2
from natsort import natsorted
import glob
import ast

def list_files_with_extension(dir_path, extension):
    '''
    列出指定目录下具有特定后缀的文件。
    
    :param dir_path: 要搜索的目录路径
    :param extension: 文件的后缀名（例如 '.txt'）
    :return: 一个包含符合条件的文件路径列表
    '''
    # 使用 glob 模块匹配通配符路径
    search_path = os.path.join(dir_path, f'*{extension}')
    files = natsorted(glob.glob(search_path))
    return files

def uniform_sample(lst, n):
    # 计算需要跳过的步长
    step = len(lst) / float(n)
    sampled_list = []
    # 使用步长进行均匀采样
    for i in range(n):
        index = int(i * step)
        sampled_list.append(lst[index])
    return sampled_list

def llm(a,b):
    prompt='Please check if the two words {} and {} are synonyms. If they are, return 1. If not, return 0,Please output the result directly'.format(a,b)

    completion = client.chat.completions.create(
        model="qwen-turbo",
        messages=[{"role": "user","content": [
            {"type": "text","text": prompt},
        ]}]
    )
    return int(completion.choices[0].message.content)

def acc_(output,label):
    right_1=0
    right_2=0
    right_3=0

    for idx,(i,j) in enumerate(zip(output,label)):
        if idx<10:
            if (i in [True,'yes'] and j in [True,'yes']) or (i in [False,'no'] and j in [False,'no']):
                right_1+=1
        
        if idx>=10 and idx<20:
            if int(i)==int(j):
                right_2+=1

        if idx>=20:
             if llm(i,j)==1:
                right_3+=1

    return right_1,right_2,right_3

import pickle
import base64

#with open('question/Question_1.pkl', 'rb') as f:
    # 使用pickle.load加载数据
 #   question = pickle.load(f)

Question_dict={
    'office0':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a table in this scene?",'yes'],
         ["Is there a cup in this scene?",'no'],
         ["Is there a carpet in this scene?",'yes'],
         ["Is there a phone in this scene?",'yes'],
         ["Is there a pen in this scene?",'no'],
         ["Is there a bag in this scene?",'yes'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many chair are there in this scene?', 2],
         ['How many table are there in this scene?', 1],
         ['How many sofa are there in this scene?', 4],
         ['How many carpet are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many toy are there in this scene?', 1],
         ['How many bag are there in this scene?', 1],
         ['How many blackboard are there in this scene?', 1],

         ['What is the closest object from the door?', 'trash can'],
         ['What is the farthest object from the trash can?', 'blackboard'],
         ['What is the closest object from the chair?', 'chair'],
         ['What is the farthest object from the door?', 'blackboard'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the sofa?', 'sofa'],
         ['What is the closest object from the table?', 'bag'],
         ['What is the farthest object from the blackboard?', 'door'],
         ['What is the farthest object from the sofa?', 'watch'],
         ['What is the closest object from the bag?', 'table']],
    
    'office1':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'no'],
         ["Is there a sofa in this scene?",'no'],
         ["Is there a book in this scene?",'yes'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'yes'],
         ["Is there a pen in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 4],
         ['How many screen are there in this scene?', 2],
         ['How many watch are there in this scene?', 1],
         ['How many towel are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many toy are there in this scene?', 0],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 1],

         ['What is the closest object from the door?', 'trash can'],
         ['What is the farthest object from the trash can?', 'pillow'],
         ['What is the closest object from the pillow?', 'pillow'],
         ['What is the farthest object from the door?', 'pillow'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the screen?', 'table'],
         ['What is the closest object from the table?', 'screen'],
         ['What is the farthest object from the blackboard?', 'screen'],
         ['What is the farthest object from the watch?', 'screen'],
         ['What is the closest object from the pen?', 'blackboard']],

    'office2':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a pen in this scene?",'no'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 5],
         ['How many screen are there in this scene?', 1],
         ['How many chair are there in this scene?', 5],
         ['How many table are there in this scene?', 3],
         ['How many watch are there in this scene?', 0],
         ['How many toy are there in this scene?', 0],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'chair'],
         ['What is the farthest object from the trash can?', 'sofa'],
         ['What is the closest object from the pillow?', 'sofa'],
         ['What is the farthest object from the door?', 'pillow'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the screen?', 'door'],
         ['What is the farthest object from the sofa?', 'chair'],
         ['What is the closest object from the sofa?', 'pillow']],

    
    'office3':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a watch in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 4],
         ['How many screen are there in this scene?', 1],
         ['How many chair are there in this scene?', 9],
         ['How many table are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many sofa are there in this scene?', 2],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'chair'],
         ['What is the farthest object from the trash can?', 'watch'],
         ['What is the closest object from the pillow?', 'sofa'],
         ['What is the farthest object from the door?', 'watch'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the screen?', 'trash can'],
         ['What is the farthest object from the sofa?', 'watch'],
         ['What is the closest object from the sofa?', 'pillow']],

    
    'office4':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'no'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'no'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a watch in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 0],
         ['How many screen are there in this scene?', 1],
         ['How many chair are there in this scene?', 12],
         ['How many table are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many sofa are there in this scene?', 0],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'trash can'],
         ['What is the farthest object from the trash can?', 'watch'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the farthest object from the door?', 'chair'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the screen?', 'door'],
         ['What is the farthest object from the watch?', 'door'],
         ['What is the closest object from the screen?', 'table']],

    'room0':
    [
         ["Is there a trash can in this scene?",'no'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a window in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a lamp in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 0],    
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 8],
         ['How many window are there in this scene?', 3],
         ['How many chair are there in this scene?', 2],
         ['How many table are there in this scene?', 1],
         ['How many watch are there in this scene?', 0],
         ['How many sofa are there in this scene?', 4],
         ['How many lamp are there in this scene?', 2],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'cabinet'],
         ['What is the farthest object from the cabinet?', 'window'],
         ['What is the closest object from the chair?', 'chair'],
         ['What is the farthest object from the door?', 'lamp'],
         ['What is the closest object from the sofa?', 'pillow'],
         ['What is the closest object from the pillow?', 'sofa'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the window?', 'door'],
         ['What is the farthest object from the lamp?', 'door'],
         ['What is the closest object from the lamp?', 'sofa']],

    
    'room1':
    [
         ["Is there a trash can in this scene?",'no'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'no'],
         ["Is there a bed in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a window in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a lamp in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 0],    
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 5],
         ['How many window are there in this scene?', 2],
         ['How many chair are there in this scene?', 0],
         ['How many bed are there in this scene?', 1],
         ['How many picture are there in this scene?', 1],
         ['How many cabinet are there in this scene?', 3],
         ['How many lamp are there in this scene?', 2],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'cabinet'],
         ['What is the farthest object from the cabinet?', 'window'],
         ['What is the closest object from the bed?', 'pillow'],
         ['What is the farthest object from the door?', 'window'],
         ['What is the closest object from the cabinet?', 'lamp'],
         ['What is the closest object from the pillow?', 'bed'],
         ['What is the closest object from the picture?', 'bed'],
         ['What is the farthest object from the window?', 'door'],
         ['What is the farthest object from the lamp?', 'door'],
         ['What is the closest object from the lamp?', 'cabinet']],
    

    'room2':
    [
         ["Is there a trash can in this scene?",'no'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a bed in this scene?",'no'],
         ["Is there a book in this scene?",'no'],
         ["Is there a table in this scene?",'yes'],
         ["Is there a window in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a lamp in this scene?",'no'],
         ["Is there a shelf in this scene?",'yes'],

         ['How many trash can are there in this scene?', 0],    
         ['How many door are there in this scene?', 1],
         ['How many shelf are there in this scene?', 1],
         ['How many window are there in this scene?', 2],
         ['How many chair are there in this scene?', 8],
         ['How many bed are there in this scene?', 0],
         ['How many picture are there in this scene?', 1],
         ['How many cabinet are there in this scene?', 0],
         ['How many lamp are there in this scene?', 0],
         ['How many vase are there in this scene?', 1],

         ['What is the closest object from the door?', 'shelf'],
         ['What is the farthest object from the shelf?', 'picture'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the door?', 'window'],
         ['What is the closest object from the shelf?', 'vase'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the picture?', 'chair'],
         ['What is the farthest object from the window?', 'door'],
         ['What is the farthest object from the picture?', 'door'],
         ['What is the farthest object from the table?', 'door']],
       
}

In [8]:
R_1,R_2,R_3=[],[],[]

In [9]:
import time
import numpy as np
import json

In [11]:
import os
from openai import OpenAI


client = OpenAI(
    # 若没有配置环境变量，请用百炼API Key将下行替换为：api_key="sk-xxx",
    api_key='',
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

In [14]:
scene

'office0'

In [15]:
for idx,scene in enumerate(list(Question_dict.keys())):
    
    #with open('/data/coding/eval/ablation/ours/3rscan_{}_3DSG.json'.format(idx+1),'r') as f:
    #    data=json.load(f)

    #with open('/data/coding/eval_replica/output/'+scene+'/'+scene+'.json','r') as f:
    #    data=json.load(f)

    data=json.load(open('/data/coding/object_graph.json'))
    scene='room0'
    
    object_list=[]

    for i in data:
        object_list.append(i)


    info=[]

    for i in object_list:     ### 计算距离
        max_distance=0
        min_distance=10000
        max_label=-1
        min_label=1
        for j in object_list:
            if i==j:
                continue
            distance=np.linalg.norm(np.array(i['bbox_center'])-np.array(j['bbox_center']))
            if distance>max_distance:
                max_distance=distance 
                max_label=j['description']
            if distance<min_distance:
                min_distance=distance
                min_label=j['description'] 
        info.append({
            'id':i['id'],
            'label':i['description'],
            #'best_view':i['color_image_idx'],
            #'max_distance':max_distance,
            #'min_distance':min_distance,
            'max_label':max_label,
            'min_label':min_label
        })

    label_list=[]         ## 唯一的label
    for i in info:
        if i['label'] not in label_list:
            label_list.append(i['label'])

    result={element: [] for element in label_list}
    for i in info:
        result[i['label']].append(i)
    
    process_prompt="This scene contains the following objects:{}".format(list(result.keys()))
    
    output_list=[]

    for iidx,j in enumerate(Question_dict[scene]):   ### 遍历每个问题
        prompt="Please refer to the following data:{}, which is in the format of \
{{'id ': object index,\
'label', Object category,\
'max_label': The category of the farthest object,\
'min_label': The category of the object closest to each other}},\
answer the following question:{} directly (in lowercase), do not output any other information, do not output punctuation marks".format(result,j[0])

        if iidx<10:
            prompt=process_prompt+"Please fully understand the meanings of these objects,than answer the following question:{},Even if there are things with similar meanings, they are correct,directly (in lowercase),\
                 do not output any other information, do not output punctuation marks".format(j[0])
        if iidx>=10 and iidx<20:
            prompt=prompt+"Please output Arabic numerals,If the object does not exist, output 0,Do not output any other information"
        if iidx>=20:
            prompt=prompt

        completion = client.chat.completions.create(
            model="qwen-max-2025-01-25", # 此处以qwen-plus为例，可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/getting-started/models
            messages=[
                {'role': 'system', 'content': 'You are a helpful assistant.'},
                {'role': 'user', 'content': prompt}],
            )
        time.sleep(1)
        print(completion.choices[0].message.content)
        output_list.append(completion.choices[0].message.content)

    print(output_list)
    output=output_list
    #output=ast.literal_eval(output)

    r1,r2,r3=acc_(output,[q[1] for q in Question_dict[scene]])

    R_1.append(r1)
    R_2.append(r2)
    R_3.append(r3)

    break



yes
no
chair
yes
no
yes
yes
no
yes
no
1
0
1
3
4
3
0
0
2
0
window
window
table
window
picture
trash can
vase
heater
chair
table
['yes', 'no', 'chair', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'no', '1', '0', '1', '3', '4', '3', '0', '0', '2', '0', 'window', 'window', 'table', 'window', 'picture', 'trash can', 'vase', 'heater', 'chair', 'table']


In [16]:
print(R_1,R_2,R_3)

[7] [4] [1]


In [12]:
for idx,scene in enumerate(list(Question_dict.keys())):
    
    #with open('/data/coding/eval/ablation/ours/3rscan_{}_3DSG.json'.format(idx+1),'r') as f:
    #    data=json.load(f)

    with open('/data/coding/eval_replica/output/'+scene+'/'+scene+'.json','r') as f:
        data=json.load(f)
    
    object_list=[]

    for i in data:
        object_list.append(i)


    info=[]

    for i in object_list:     ### 计算距离
        max_distance=0
        min_distance=10000
        max_label=-1
        min_label=1
        for j in object_list:
            if i==j:
                continue
            distance=np.linalg.norm(np.array(i['bbox_center'])-np.array(j['bbox_center']))
            if distance>max_distance:
                max_distance=distance 
                max_label=j['description']
            if distance<min_distance:
                min_distance=distance
                min_label=j['description'] 
        info.append({
            'id':i['id'],
            'label':i['description'],
            #'best_view':i['color_image_idx'],
            #'max_distance':max_distance,
            #'min_distance':min_distance,
            'max_label':max_label,
            'min_label':min_label
        })

    label_list=[]         ## 唯一的label
    for i in info:
        if i['label'] not in label_list:
            label_list.append(i['label'])

    result={element: [] for element in label_list}
    for i in info:
        result[i['label']].append(i)
    
    process_prompt="This scene contains the following objects:{}".format(list(result.keys()))
    
    output_list=[]

    for iidx,j in enumerate(Question_dict[scene]):   ### 遍历每个问题
        prompt="Please refer to the following data:{}, which is in the format of \
{{'id ': object index,\
'label', Object category,\
'max_label': The category of the farthest object,\
'min_label': The category of the object closest to each other}},\
answer the following question:{} directly (in lowercase), do not output any other information, do not output punctuation marks".format(result,j[0])

        if iidx<10:
            prompt=process_prompt+"Please fully understand the meanings of these objects,than answer the following question:{},Even if there are things with similar meanings, they are correct,directly (in lowercase),\
                 do not output any other information, do not output punctuation marks".format(j[0])
        if iidx>=10 and iidx<20:
            prompt=prompt+"Please output Arabic numerals,If the object does not exist, output 0,Do not output any other information"
        if iidx>=20:
            prompt=prompt

        completion = client.chat.completions.create(
            model="qwen-max-2025-01-25", # 此处以qwen-plus为例，可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/getting-started/models
            messages=[
                {'role': 'system', 'content': 'You are a helpful assistant.'},
                {'role': 'user', 'content': prompt}],
            )
        time.sleep(1)
        print(completion.choices[0].message.content)
        output_list.append(completion.choices[0].message.content)

    print(output_list)
    output=output_list
    #output=ast.literal_eval(output)

    r1,r2,r3=acc_(output,[q[1] for q in Question_dict[scene]])

    R_1.append(r1)
    R_2.append(r2)
    R_3.append(r3)



yes
yes
yes
yes
yes
no
yes
no
no
no
2
3
2
1
3
0
0
0
0
0
door
door
chair
cloud
rug
tissue box
folder
cloud
door
folder
['yes', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'no', '2', '3', '2', '1', '3', '0', '0', '0', '0', '0', 'door', 'door', 'chair', 'cloud', 'rug', 'tissue box', 'folder', 'cloud', 'door', 'folder']
yes
yes
yes
no
no
yes
yes
yes
no
no
1
2
8
0
0
1
0
0
0
0
mirror
door
box
trash can
table
remote
phone
door
door
pillow
['yes', 'yes', 'yes', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'no', '1', '2', '8', '0', '0', '1', '0', '0', '0', '0', 'mirror', 'door', 'box', 'trash can', 'table', 'remote', 'phone', 'door', 'door', 'pillow']
no
yes
yes
yes
yes
yes
yes
no
no
no
0
1
3
0
3
5
0
0
0
0
wallet
door
table
couch
table
table
desk
door
door
ottoman
['no', 'yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'no', 'no', '0', '1', '3', '0', '3', '5', '0', '0', '0', '0', 'wallet', 'door', 'table', 'couch', 'table', 'table', 'desk', 'door', 'door', 'ottoman']
yes
yes
yes
yes
no
no
yes

In [10]:
scene

'office1'

In [10]:
result

{'mattress': [{'id': 0,
   'label': 'mattress',
   'max_label': 'cabinet',
   'min_label': 'curtain'}],
 'rug': [{'id': 1,
   'label': 'rug',
   'max_label': 'cabinet',
   'min_label': 'laptop'},
  {'id': 8, 'label': 'rug', 'max_label': 'chair', 'min_label': 'cabinet'}],
 'laptop': [{'id': 2,
   'label': 'laptop',
   'max_label': 'cabinet',
   'min_label': 'pillow'}],
 'pillow': [{'id': 3,
   'label': 'pillow',
   'max_label': 'cabinet',
   'min_label': 'laptop'}],
 'glove': [{'id': 4,
   'label': 'glove',
   'max_label': 'cabinet',
   'min_label': 'laptop'}],
 'book': [{'id': 5,
   'label': 'book',
   'max_label': 'cabinet',
   'min_label': 'curtain'}],
 'drawer': [{'id': 6,
   'label': 'drawer',
   'max_label': 'cabinet',
   'min_label': 'picture'}],
 'picture': [{'id': 7,
   'label': 'picture',
   'max_label': 'glove',
   'min_label': 'picture'},
  {'id': 10,
   'label': 'picture',
   'max_label': 'glove',
   'min_label': 'picture'}],
 'cabinet': [{'id': 9,
   'label': 'cabinet',
  

In [15]:
print(sum(R_1[2:])/80)
print(sum(R_2[2:])/80)
print(sum(R_3[2:])/80)

print(sum(R_1[2:]+R_2[2:]+R_3[2:])/240)

0.8375
0.3875
0.125
0.45


In [17]:
print(R_1[2:])
print(R_2[2:])
print(R_3[2:])

[8, 7, 8, 8, 9, 9, 9, 9]
[3, 3, 5, 2, 5, 4, 5, 4]
[1, 0, 2, 0, 4, 2, 0, 1]


In [19]:
question

{'6e67e550-1209-2cd0-8294-7cc2564cf82c': [['Is there a lamp in this scene?',
   'yes'],
  ['Is there a commode in this scene?', 'yes'],
  ['Is there a car in this scene?', 'no'],
  ['Is there a curtain in this scene?', 'yes'],
  ['Is there a door in this scene?', 'yes'],
  ['Is there a tv stand in this scene?', 'yes'],
  ['Is there a ceiling in this scene?', 'yes'],
  ['Is there a car in this scene?', 'no'],
  ['Is there a pillow in this scene?', 'yes'],
  ['Is there a cabinet in this scene?', 'yes'],
  ['How many wall are there in this scene?', 3],
  ['How many ceiling are there in this scene?', 1],
  ['How many commode are there in this scene?', 1],
  ['How many floor are there in this scene?', 1],
  ['How many door are there in this scene?', 1],
  ['How many cabinet are there in this scene?', 4],
  ['How many shelf are there in this scene?', 1],
  ['How many pillow are there in this scene?', 3],
  ['How many tv stand are there in this scene?', 1],
  ['How many picture are there in t

In [13]:
info

[{'id': 0,
  'label': 'mattress',
  'max_label': 'cabinet',
  'min_label': 'curtain'},
 {'id': 1, 'label': 'rug', 'max_label': 'cabinet', 'min_label': 'laptop'},
 {'id': 2, 'label': 'laptop', 'max_label': 'cabinet', 'min_label': 'pillow'},
 {'id': 3, 'label': 'pillow', 'max_label': 'cabinet', 'min_label': 'laptop'},
 {'id': 4, 'label': 'glove', 'max_label': 'cabinet', 'min_label': 'laptop'},
 {'id': 5, 'label': 'book', 'max_label': 'cabinet', 'min_label': 'curtain'},
 {'id': 6, 'label': 'drawer', 'max_label': 'cabinet', 'min_label': 'picture'},
 {'id': 7, 'label': 'picture', 'max_label': 'glove', 'min_label': 'picture'},
 {'id': 8, 'label': 'rug', 'max_label': 'chair', 'min_label': 'cabinet'},
 {'id': 9, 'label': 'cabinet', 'max_label': 'glove', 'min_label': 'rug'},
 {'id': 10, 'label': 'picture', 'max_label': 'glove', 'min_label': 'picture'},
 {'id': 11, 'label': 'chair', 'max_label': 'cabinet', 'min_label': 'drawer'},
 {'id': 12,
  'label': 'curtain',
  'max_label': 'cabinet',
  'min